# 4. Nonanes and vaporisation enthalpies: pre-registered test, search boxes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Singati2/LOYOLA-PAPER/blob/revision/v40.31-scientific/colab/04_nonane_vaporisation_enthalpies.ipynb)

Runs these scripts in this order (about 101 minutes on the lab machine):

1. `external/baselines.py` (lab: 5 min)
2. `external/baselines.py --dataset nonane` (lab: 5 min)
3. `external/nonane_validation.py` (lab: 14 min)
4. `external/transfer_exploratory.py` (lab: 2 min)
5. `external/uncertainty_exploratory.py` (lab: 14 min)
6. `external/wide_box_diagnostics.py` (lab: 3 min)
7. `external/box_sensitivity.py` (lab: 45 min)
8. `external/nonane_tb_conflict.py` (lab: 11 min)
9. `external/make_box_table.py` (lab: 1 min)
10. `external/make_v37_table.py` (lab: 1 min)

Each script rewrites its committed output files; notebook 11 compares them with the commit. If the session disconnects, open this notebook again and run all cells: finished scripts are skipped and the run continues with the next one. Keep the browser tab open while it runs.

In [ ]:
# Setup (the same cell in every notebook of colab/; about 2 minutes the first time in a session)
REVISION = "revision/v40.31-scientific"   # the branch or tag to run; every notebook of one run must use the same commit
REPO_URL = "https://github.com/Singati2/LOYOLA-PAPER.git"
import os, sys, shutil, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")             # progress and outputs are kept in MyDrive/LOYOLA_colab_run
    WORK = "/content/LOYOLA-PAPER"
    if not os.path.exists(WORK):
        token = None
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")  # Colab secret (key icon on the left), needed while the repository is private
        except Exception:
            pass
        bundle = "/content/drive/MyDrive/LOYOLA-PAPER.bundle"
        if token:
            url = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
            subprocess.run(["git", "clone", "--quiet", "--branch", REVISION, url, WORK], check=True)
            subprocess.run(["git", "-C", WORK, "remote", "set-url", "origin", REPO_URL], check=True)   # token not kept on disk
        elif os.path.exists(bundle):
            subprocess.run(["git", "clone", "--quiet", "--branch", REVISION, bundle, WORK], check=True)
        else:
            subprocess.run(["git", "clone", "--quiet", "--branch", REVISION, REPO_URL, WORK], check=True)
    os.chdir(WORK)
    ENV = "/content/loyola-env"
else:                                         # run locally from inside the repository
    while not os.path.exists("verify.py") and os.getcwd() != "/":
        os.chdir("..")
    ENV = os.path.join(os.getcwd(), ".venv-colab")
PY = os.path.join(ENV, "bin", "python")
if not os.path.exists(PY):                    # Python 3.13 and the pinned versions that produced the committed outputs
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)
    UV = ["uv"] if shutil.which("uv") else [sys.executable, "-m", "uv"]
    subprocess.run(UV + ["venv", "--quiet", "--python", "3.13", ENV], check=True)
    subprocess.run(UV + ["pip", "install", "--quiet", "--python", PY, "-r", "requirements-lock.txt",
                         "-r", "requirements-extras-lock.txt"], check=True)
print(subprocess.run([PY, "-c", "import sys, numpy, scipy; print('python', sys.version.split()[0], '| numpy', "
                      "numpy.__version__, '| scipy', scipy.__version__)"], capture_output=True, text=True).stdout.strip())
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
STATE = os.environ.get("LOYOLA_STATE") or ("/content/drive/MyDrive/LOYOLA_colab_run" if IN_COLAB
                                           else os.path.join(os.getcwd(), "colab_run"))   # as in pipeline.py


def pipeline(*args):
    """Run colab/pipeline.py with the pinned Python, show its output as it comes, stop on failure."""
    p = subprocess.Popen([PY, "-u", "colab/pipeline.py", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError("pipeline.py " + " ".join(args) + " reported a problem (see the output above)")


In [ ]:
pipeline("--stage", "4")